# LintGate Full Analysis

**Auto-generated** comprehensive project analysis notebook.
Runs the entire LintGate analysis pipeline offline and produces
a portable JSON artifact with a prioritized action plan.

- **Target repo**: `https://github.com/rohanvinaik/LintGate.git`
- **Branch**: `prescriptive-spec-system`
- **Source dirs**: `['lintgate', 'mcp_tools']`
- **Mutation profiling**: `True`
- **Mode**: Self-analysis (LintGate analyzing itself)

**How to use:** Runtime > Run all. Download the zip at the end.
Pass the `action_plan.json` to any LLM coding agent.

The action plan is:
- **Prioritized**: P0 blocking → P1 critical → P2 important → P3 improve
- **Dependency-ordered**: lint fixes before spec work, tests before mutation profiling
- **Actionable**: each item has specific tool commands and expected outcomes


## Step 1: Install LintGate + Clone Target


In [ ]:
import importlib, os, shutil, subprocess, sys

REPO_URL = "https://github.com/rohanvinaik/LintGate.git"
BRANCH = "prescriptive-spec-system"
PROJECT_DIR = "/content/project"
SRC_DIRS = ['lintgate', 'mcp_tools']

# Uncomment if repo is private:
# GITHUB_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

clone_url = REPO_URL
try:
    GITHUB_TOKEN
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    print('Using authenticated clone')
except NameError:
    print('Using public clone')

result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, clone_url, PROJECT_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f'Clone failed: {result.stderr}')
print(f'Cloned to {PROJECT_DIR}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'hatchling', 'pyyaml', 'packaging'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', PROJECT_DIR], capture_output=True, text=True)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
for m in list(sys.modules):
    if any(m.startswith(d.split('/')[0]) for d in SRC_DIRS):
        del sys.modules[m]
importlib.invalidate_caches()
print('Self-analysis mode: LintGate is both tool and target')


## Step 2: Run Full Analysis


In [ ]:
import json, os, time

INCLUDE_MUTATION = True
MUTATION_BUDGET_MS = 500

from lintgate.offline_analysis import run_full_analysis

print('Starting comprehensive analysis...')
print('This runs: lint + specification + composition + performance + test coverage')
if INCLUDE_MUTATION:
    print('+ mutation profiling (this is the slow part)')
print()

result = run_full_analysis(
    PROJECT_DIR,
    src_dirs=['lintgate', 'mcp_tools'],
    include_mutation=INCLUDE_MUTATION,
    mutation_budget_ms=MUTATION_BUDGET_MS,
)

print(f'Analysis complete in {result["elapsed_s"]}s')
print(f'Source files: {result["project"]["total_source_files"]}')
print(f'Test files: {result["project"]["total_test_files"]}')
print(f'Total LoC: {result["project"]["total_loc"]}')
print(f'Lint findings: {result["lint"]["total_findings"]} ({result["lint"]["auto_fixable"]} auto-fixable)')
print(f'Functions analyzed: {result["specification"]["total_functions"]}')
print(f'Under-specified: {result["specification"]["under_specified_count"]}')
print(f'Action items: {len(result["action_plan"])}')


## Step 3: Review Action Plan


In [ ]:
print('=' * 80)
print('ACTION PLAN — prioritized fixes for LLM implementation')
print('=' * 80)

for action in result['action_plan']:
    rank = action['rank']
    priority = action['priority']
    cat = action['category']
    f = action.get('file', '')
    deps = action.get('depends_on', [])
    effort = action.get('estimated_effort', '')
    print(f'\n[{rank}] {priority} | {cat} | {effort}')
    print(f'    File: {f}')
    if action.get('function'):
        print(f'    Function: {action["function"]}')
    print(f'    Action: {action["action"]}')
    if deps:
        print(f'    Depends on: {deps}')

print(f'\n{"=" * 80}')
p_counts = {}
for a in result['action_plan']:
    p = a['priority']
    p_counts[p] = p_counts.get(p, 0) + 1
for p, n in sorted(p_counts.items()):
    print(f'  {p}: {n}')


## Step 4: Save & Download


In [ ]:
import json, os

# Save the full analysis artifact
output_path = os.path.join(PROJECT_DIR, '.lintgate', 'full_analysis.json')
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(result, f, indent=2)
size_kb = os.path.getsize(output_path) / 1024
print(f'Analysis saved: {output_path} ({size_kb:.0f} KB)')

# Also save a compact action-plan-only file
plan_path = os.path.join(PROJECT_DIR, '.lintgate', 'action_plan.json')
with open(plan_path, 'w') as f:
    json.dump({
        'project': result['project']['name'],
        'timestamp': result['timestamp'],
        'summary': {
            'source_files': result['project']['total_source_files'],
            'lint_findings': result['lint']['total_findings'],
            'under_specified': result['specification']['under_specified_count'],
            'action_count': len(result['action_plan']),
        },
        'action_plan': result['action_plan'],
    }, f, indent=2)
print(f'Action plan saved: {plan_path}')

# Zip for download
zip_path = '/content/lintgate_analysis.zip'
!cd {PROJECT_DIR} && zip -r {zip_path} .lintgate/full_analysis.json .lintgate/action_plan.json -x '*.DS_Store'

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / 1024 / 1024
    print(f'\nDownload ready: {size_mb:.1f} MB')
    try:
        from google.colab import files
        files.download(zip_path)
        print('Download started!')
    except ImportError:
        print(f'Not in Colab. Results at: {zip_path}')
else:
    print('No zip created.')

print(f'\n--- HOW TO USE ---')
print(f'Pass the action_plan.json to your LLM coding agent with:')
print(f'  "Implement the fixes in this action plan, starting from rank 1."')
print(f'  "Each action has priority, dependencies, and specific instructions."')
print(f'  "Apply results to: /Users/rohanvinaik/tools/lintgate"')
